# Loading and exploring spectral grids

This notebook builds a tiny **synthetic, non-scientific** SPHINX-format cache in a temporary directory. Its only purpose is to exercise the public `SpectralGrid` workflow offline; do not use these analytic flux arrays for science. A real SPHINX workflow starts with `download_sphinx_grid()` and uses the same retrieval calls.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

from speclib import SpectralGrid, Spectrum
from speclib.utils import set_library_root

## Make an isolated teaching cache

The fixture has a complete 2 x 2 x 2 cube in effective temperature, surface gravity, and the filename's `logZ` metallicity field. C/O is fixed at 0.5, just as it must be for one real SPHINX `SpectralGrid`.

In [ ]:
temporary_cache = TemporaryDirectory()
library_root = Path(temporary_cache.name)
sphinx_dir = library_root / "sphinx"
sphinx_dir.mkdir()

wavelength_um = np.linspace(1.0, 2.0, 251)
shape = 1.0 + 0.08 * np.sin(8 * np.pi * wavelength_um)
for teff in (3000.0, 3100.0):
    for logg in (4.0, 4.5):
        for logz in (0.0, 0.25):
            scale = 1e8 * (1 + (teff - 3000) / 1000 + 0.1 * (logg - 4) + 0.2 * logz)
            flux = scale * shape
            filename = (
                f"Teff_{teff:.1f}_logg_{logg:.2f}_logZ_{logz:+.2f}_CtoO_0.5.txt"
            )
            np.savetxt(
                sphinx_dir / filename,
                np.column_stack((wavelength_um, flux)),
                header="Wavelength [um]  F_lambda [W/m2/m] -- SYNTHETIC TEACHING DATA",
            )

set_library_root(library_root)

## Load and inspect the grid

The requested temperature bounds extend beyond the fixture. `SpectralGrid` clips them to available axis values and emits a `UserWarning`. Interior off-grid bounds would instead be expanded outward to bracketing grid values.

In [ ]:
grid = SpectralGrid(
    teff_bds=(2950, 3150),
    logg_bds=(4.0, 4.5),
    feh_bds=(0.0, 0.25),
    model_grid="sphinx",
    co_ratio=0.5,
)
print("aligned bounds:", grid.teff_bds, grid.logg_bds, grid.feh_bds)
print("loaded points:", grid.points)
print("data shape:", grid.data.shape)
print("wavelength unit:", grid.wavelength.unit, "flux unit:", grid.unit)

## Exact, nearest, and interpolated flux

`get_flux` always returns a one-dimensional quantity aligned with `grid.wavelength`. With `interpolate=False`, an off-grid SPHINX request selects the nearest *actual combination*. With `interpolate=True`, all necessary corners must exist.

In [ ]:
exact = grid.get_flux(3000, 4.0, 0.0, interpolate=True)
nearest = grid.get_flux(3040, 4.1, 0.05, interpolate=False)
interpolated = grid.get_flux(3050, 4.25, 0.125, interpolate=True)
print(exact.shape, nearest.shape, interpolated.shape)
print("nearest equals lower corner:", np.allclose(nearest.value, exact.value))
print("interpolation differs:", not np.allclose(interpolated.value, exact.value))

In [ ]:
interpolated_spectrum = Spectrum(
    spectral_axis=grid.wavelength,
    flux=interpolated,
)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(grid.wavelength.to_value(u.um), exact.value, label="exact corner")
ax.plot(grid.wavelength.to_value(u.um), interpolated_spectrum.flux.value, label="trilinear midpoint")
ax.set(xlabel="Wavelength [micrometer]", ylabel=f"Flux density [{grid.unit}]")
ax.legend();

## Retrieval boundaries

Constructor clipping is not extrapolation. A request outside the loaded bounds raises `ValueError`. Real SPHINX slices can also lack a required interpolation corner; that condition raises `ValueError` rather than silently filling the hole.

In [ ]:
try:
    grid.get_flux(3200, 4.25, 0.125)
except ValueError as error:
    print(error)

In [ ]:
set_library_root(None)
temporary_cache.cleanup()